# Ciringoni M3 E5 P2 dan E6 augmentasi

Latih kontrol YOLO26n baru, lalu uji head P2 dan penonaktifan mosaic secara terpisah. Semua run memakai sumber citra olahan 640 yang sama. Penilaian dan analisis kesalahan memakai 356 gambar validation dan 2.205 anotasi acuan.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZipFile

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
OUTPUT = WORK / "m3"
OUTPUT.mkdir()
REPO = "https://github.com/Jordyee/ciringoni-data-mining.git"
SOURCE = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
PROJECT = Path("/tmp/ciringoni-source/Data_Mining_Ciringoni_M1")
SCRIPTS = PROJECT / "hasil_m2"

def execute(*args):
    subprocess.run([str(arg) for arg in args], check=True, cwd=PROJECT)

def input_file(name):
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f"Expected one mounted {name}, found {len(paths)}"
    return paths[0]

subprocess.run(["git", "clone", "--quiet", REPO, str(Path("/tmp/ciringoni-source"))], check=True)
execute("git", "checkout", "--quiet", SOURCE)
execute(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210", "pycocotools==2.0.11")
archives = list(INPUT.rglob("processed_train_val.zip"))
assert len(archives) <= 1
ROOT = Path("/tmp/ciringoni-processed/dataset")
if archives:
    with ZipFile(archives[0]) as archive:
        archive.extractall(ROOT.parent)
else:
    inputs = [path.parent.parent for path in INPUT.rglob("train")
              if path.is_dir() and path.parent.name == "images" and (path.parent.parent / "labels/val").is_dir()]
    assert len(inputs) == 1, f"Expected one processed input root, found {len(inputs)}"
    shutil.copytree(inputs[0], ROOT)
for split, expected in (("train", 1244), ("val", 356)):
    images = {path.stem for path in (ROOT / "images" / split).glob("*.jpg")}
    labels = {path.stem for path in (ROOT / "labels" / split).glob("*.txt")}
    assert len(images) == expected and images == labels
    print(f"{split}: {len(images)} JPG and labels")
assert not (ROOT / "images/test").exists()
YAML = ROOT / "data.yaml"
YAML.write_text(json.dumps({"path": str(ROOT), "train": "images/train", "val": "images/val", "names": ["infected"]}), encoding="utf-8")
for target, name in [(PROJECT / "data", "yolo26n.pt"), (PROJECT / "hasil_m1", "coco_val_gt.json")]:
    target.mkdir(exist_ok=True)
    shutil.copyfile(input_file(name), target / name)
with (PROJECT / "data/yolo26n.pt").open("rb") as stream:
    WEIGHTS_SHA = hashlib.file_digest(stream, "sha256").hexdigest()
assert WEIGHTS_SHA == "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"

cpus = sorted(os.sched_getaffinity(0))[:4]
assert len(cpus) == 4
os.sched_setaffinity(0, cpus)
os.environ.update(OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
trainer_file = SCRIPTS / "train_mosaic.py"
trainer_source = trainer_file.read_text()
required = '    if not torch.cuda.is_available():\n        raise SystemExit("CUDA is required for this experiment.")\n'
assert required in trainer_source
trainer_source = trainer_source.replace(required, '    torch.set_num_threads(4)\n')
trainer_source = trainer_source.replace("{torch.cuda.get_device_name(0)}", "Kaggle CPU")
trainer_source = trainer_source.replace("device=0, workers=0", 'device="cpu", workers=0')
trainer_source = trainer_source.replace("        torch.cuda.empty_cache()\n", "")
compile(trainer_source, str(trainer_file), "exec")
trainer_file.write_text(trainer_source)


## Training terkontrol

Lima epoch, batch 8, seed 42. Simpan `args.yaml`, `results.csv`, kurva, dan checkpoint tiap run.


In [ ]:
import re
import torch, ultralytics

RUNS = ("E0_640", "E5_640_p2", "E6_640_no_mosaic")
EPOCHS = 5
TRAINING = {}
print("Training:", "Kaggle CPU", torch.__version__, ultralytics.__version__)
for run in RUNS:
    cmd = [sys.executable, SCRIPTS / "train_mosaic.py", "--data", YAML, "--epochs", EPOCHS, "--run", run]
    log_path = OUTPUT / f"{run}.log"
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
        [str(arg) for arg in cmd], cwd=PROJECT, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
    ) as trainer:
        for line in trainer.stdout:
            log.write(line)
            if any(marker in line for marker in ("training started", "Transferred ", "training finished", "Error")):
                print(line.strip(), flush=True)
        if trainer.wait() != 0:
            log.flush()
            raise RuntimeError(f"Training failure for {run}:\n" + log_path.read_text(encoding="utf-8")[-6000:])
    directory = SCRIPTS / "runs" / run
    lines = (directory / "results.csv").read_text().splitlines()
    checkpoint = directory / "weights/best.pt"
    log_text = log_path.read_text(encoding="utf-8")
    assert len(lines) == EPOCHS + 1 and checkpoint.stat().st_size > 0
    assert not re.search(r"reducing batch|CUDA out of memory", log_text, re.IGNORECASE)
    transfers = re.findall(r"Transferred (\d+)/(\d+) items from pretrained weights", log_text)
    assert transfers, f"Missing transfer evidence for {run}"
    with checkpoint.open("rb") as stream:
        TRAINING[run] = {"epochs_completed": EPOCHS,
                         "checkpoint_sha256": hashlib.file_digest(stream, "sha256").hexdigest(),
                         "weights_transfer": [int(n) for n in transfers[-1]]}
    shutil.copytree(directory, OUTPUT / "runs" / run)


## Metrik validation dan latensi CPU

AP-small memakai COCOeval. mAP50 dan mAP50–95 di CSV berasal dari validator Ultralytics. Benchmark PyTorch pada CPU memakai empat thread dan memanaskan 10 gambar. Latensi menjumlahkan waktu preproses, inferensi, dan pascaproses dari Ultralytics.


In [ ]:
import pandas as pd
execute(sys.executable, SCRIPTS / "evaluasi_mosaic.py", YAML, "--device", "cpu", "--runs", *RUNS)
execute(sys.executable, SCRIPTS / "evaluasi_mosaic.py", YAML, "--runs", *RUNS, "--cpu-benchmark")
for run in RUNS:
    shutil.copytree(SCRIPTS / "evaluasi" / run, OUTPUT / "evaluation" / run)
METRICS = OUTPUT / "eksperimen.csv"
shutil.copyfile(SCRIPTS / "evaluasi/eksperimen_mosaic_val.csv", METRICS)
rows = pd.read_csv(METRICS).set_index("id").loc[list(RUNS)]
assert rows["gambar"].eq(356).all()
display(rows[["AP-small", "mAP50", "mAP50-95", "presisi_conf_0.05_iou_0.5", "recall_conf_0.05_iou_0.5", "cpu_median_ms", "cpu_p95_ms"]])


## Analisis kesalahan dan keputusan ablation

Confidence 0,05 dan IoU 0,50. Hitung objek terlewat untuk kotak dengan sisi di bawah 8 piksel secara terpisah dari definisi AP-small. Pilihan di sini berlaku untuk ablation gambar olahan 640.


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

GT = COCO(str(PROJECT / "hasil_m1/coco_val_gt.json"))
assert len(GT.imgs) == 356 and len(GT.anns) == 2205
errors = []
predictions_by_run = {}
by_image = {}
for run in RUNS:
    predictions = json.loads((OUTPUT / "evaluation" / run / "coco_val_predictions.json").read_text())
    retained = [item for item in predictions if item["score"] >= 0.05]
    predictions_by_run[run] = defaultdict(list)
    for item in retained:
        predictions_by_run[run][item["image_id"]].append(item)
    if retained:
        scorer = COCOeval(GT, GT.loadRes(retained), "bbox")
        scorer.params.iouThrs = np.array([0.50])
        scorer.params.areaRng = [scorer.params.areaRng[0]]
        scorer.params.areaRngLbl = ["all"]
        scorer.evaluate()
        matched_gt = {int(value) for item in scorer.evalImgs if item is not None
                      for value in item["dtMatches"][0][~item["dtIgnore"][0]] if value}
    else:
        matched_gt = set()
    images = defaultdict(lambda: [0, 0])
    for ann in GT.anns.values():
        subset = int(min(ann["bbox"][2:]) >= 8)
        images[ann["image_id"]][subset] += int(ann["id"] not in matched_gt)
    by_image[run] = images
    small = {ann["id"] for ann in GT.anns.values() if min(ann["bbox"][2:]) < 8}
    tp = len(matched_gt)
    errors.append({"run": run, "tp": tp, "fp": len(retained) - tp, "fn": len(GT.anns) - tp,
                   "gt_short_side_lt_8px": len(small), "tp_short_side_lt_8px": len(small & matched_gt)})
errors = pd.DataFrame(errors)
errors.to_csv(OUTPUT / "analisis_kesalahan.csv", index=False)
display(errors)

control = rows.loc["E0_640"]
mbr = "recall_conf_0.05_iou_0.5"
choices = []
for run in RUNS[1:]:
    row = rows.loc[run]
    change = {"AP-small": float(row["AP-small"] - control["AP-small"]),
              "mAP50-95": float(row["mAP50-95"] - control["mAP50-95"]),
              "recall": float(row[mbr] - control[mbr])}
    accepted = change["AP-small"] >= 0.005 and change["mAP50-95"] >= -0.005 and change["recall"] >= -0.02
    choices.append({"run": run, "delta_vs_E0_640": change, "eligible": accepted})
eligible = [choice["run"] for choice in choices if choice["eligible"]]
if eligible:
    top_ap = max(rows.loc[run, "AP-small"] for run in eligible)
    near = [run for run in eligible if top_ap - rows.loc[run, "AP-small"] < 0.005]
    selection = min(near, key=lambda run: rows.loc[run, "cpu_p95_ms"])
else:
    selection = "E0_640"
metadata = {"source_commit": SOURCE, "initial_weights_sha256": WEIGHTS_SHA, "train_images": 1244, "validation_images": 356, "validation_boxes": 2205,
            "epochs": EPOCHS, "batch": 8, "seed": 42, "device": "Kaggle CPU",
            "torch": str(torch.__version__), "ultralytics": ultralytics.__version__,
            "training": TRAINING, "matched_control": "E0_640", "choices": choices, "selected_m3_640": selection}
(OUTPUT / "manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))

image_id = max(GT.imgs, key=lambda index: (by_image["E0_640"][index][0], by_image["E0_640"][index][1], -index))
image = Image.open(ROOT / "images/val" / GT.imgs[image_id]["file_name"])
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for axis, title in zip(axes, ("Anotasi acuan", *RUNS), strict=True):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
    detections = GT.loadAnns(GT.getAnnIds(imgIds=[image_id])) if title == "Anotasi acuan" else predictions_by_run[title][image_id]
    for item in detections:
        x, y, w, h = item["bbox"]
        axis.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.1,
                                 edgecolor="lime" if title == "Anotasi acuan" else "orange"))
fig.savefig(OUTPUT / "contoh_kesalahan_val.png", dpi=140)
plt.show()
